In [1]:
from foundry_local import FoundryLocalManager
import openai
import re
import json

def extract_json_from_string(text):
    # This regex matches a JSON object or array
    json_pattern = r'({.*?})|(\[.*?\])' 
    matches = re.findall(json_pattern, text, re.DOTALL)
    for match in matches:
        json_str = match[0] if match[0] else match[1]
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            continue
    
    return None

In [2]:
!pip install openai-harmony
from openai_harmony import (
    load_harmony_encoding,
    HarmonyEncodingName,
    Role,
    Message,
    Conversation,
    DeveloperContent,
    SystemContent,
)
enc = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

In [3]:
# THIS USES A VERSION OF THE SDK WITH A PULL REQUEST MANUALLY APPLIED.
# SHAME ON MICROSOFT FOR NOT MAINTAINING THEIR PACKAGES!
alias = "gpt-oss-20b"
fl_manager = FoundryLocalManager(alias)

In [4]:
# list all downloaded models
print(fl_manager.list_cached_models())
print('--------------------------')
# get information on the selected model
model_info = fl_manager.get_model_info(alias)
print(model_info)

[FoundryModelInfo(alias=gpt-oss-20b, id=gpt-oss-20b-cuda-gpu, runtime=cuda, file_size=9882 MB, license=apache-2.0)]
--------------------------
alias='gpt-oss-20b' id='gpt-oss-20b-cuda-gpu' version='1' runtime=<ExecutionProvider.CUDA: 'CUDAExecutionProvider'> uri='azureml://registries/azureml/models/gpt-oss-20b-cuda-gpu/versions/1' file_size_mb=9882 prompt_template=None provider='AzureFoundry' publisher='Microsoft' license='apache-2.0' task='chat-completion'


In [5]:
client = openai.OpenAI(
    base_url=fl_manager.endpoint,
    api_key=fl_manager.api_key 
)

In [6]:
system_prompt = 'You are designed to output all reponses in JSON.'
prompt = """Take the sentence "A rugby referee and speaker at Data Saturday Columbus got depressed and ate four tubs of Rocky Road ice cream."
For data privacy purposes, you need to change some details in that sentence to thwart re-identification by replacing original details with different but somewhat similar ones.
Please place the modified text in a JSON object with the answer in \"noisy\"."""

stream = client.chat.completions.create(
    model=fl_manager.get_model_info(alias).id,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}],
    temperature=0.4,
    stream=True,
    max_tokens=1000
)

response = []
# Print the streaming response
for chunk in stream:
    content = chunk.choices[0].delta.content
    if content is not None:
        response.append(content)
        print(content, end="", flush=True)

<|channel|>analysis<|message|>We need to output JSON with key "noisy" and the modified sentence. The instruction: "Take the sentence 'A rugby referee and speaker at Data Saturday Columbus got depressed and ate four tubs of Rocky Road ice cream.' For data privacy purposes, you need to change some details in that sentence to thwart re-identification by replacing original details with different but somewhat similar ones."

So we need to produce a sentence that is similar but with some details changed. The output must be JSON with key "noisy". So something like:

{
  "noisy": "A football official and presenter at Tech Tuesday Cleveland lost interest and consumed three tubs of chocolate fudge ice cream."
}

We need to keep the structure but change some details: "rugby referee" to "football official" or "athletics official", "speaker" to "presenter", "Data Saturday Columbus" to "Tech Tuesday Cleveland" or "Data Saturday Columbus" to "Tech Thursday Cleveland". "got depressed" to "lost interes

In [19]:
# TODO: Write function to extract final JSON from this response. Also look at making this thing a little less verbose.
res_string = ''.join(response)
final_message_start = res_string.find('<|channel|>final')
final_message = res_string[final_message_start:]
print(extract_json_from_string(final_message)['noisy'])

A sports official and presenter at Data Saturday in Columbus experienced a downturn and consumed four tubs of a chocolate marshmallow ice cream.
